In [ ]:
dbutils.widgets.text("catalog", "datasheet_project")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS datasheet_project.gold")
print("✅ Gold schema ready")

In [0]:
%sql
CREATE OR REPLACE TABLE gold.chip_catalog AS
SELECT
  -- Identity
  file_name,
  chip_name,
  manufacturer,
  
  -- Normalize architecture into a clean category
  CASE
    WHEN LOWER(architecture) LIKE '%cortex-m0%' THEN 'ARM Cortex-M0'
    WHEN LOWER(architecture) LIKE '%cortex-m3%' THEN 'ARM Cortex-M3'
    WHEN LOWER(architecture) LIKE '%cortex-m4%' THEN 'ARM Cortex-M4'
    WHEN LOWER(architecture) LIKE '%cortex-m7%' THEN 'ARM Cortex-M7'
    WHEN LOWER(architecture) LIKE '%avr%'       THEN 'AVR 8-bit'
    WHEN LOWER(architecture) LIKE '%xtensa%'    THEN 'Xtensa 32-bit'
    WHEN LOWER(architecture) LIKE '%esp8266%'   THEN 'Xtensa 32-bit'
    WHEN LOWER(architecture) LIKE '%16-bit%'    THEN '16-bit RISC'
    WHEN LOWER(architecture) LIKE '%pic%'       THEN 'PIC RISC'
    WHEN LOWER(architecture) LIKE '%risc%'      THEN 'Generic RISC'
    WHEN architecture IS NULL                    THEN 'Unknown'
    ELSE 'Other'
  END AS architecture_family,
  
  architecture AS architecture_raw,
  
  -- Core specs
  flash_kb,
  ram_kb,
  max_clock_mhz,
  pin_count,
  
  -- Derived: voltage range
  voltage_min,
  voltage_max,
  ROUND(voltage_max - voltage_min, 2) AS voltage_range,
  
  -- Derived: temp range
  operating_temp_min,
  operating_temp_max,
  (operating_temp_max - operating_temp_min) AS temp_range,
  
  -- Flags
  CASE WHEN flash_kb IS NULL OR ram_kb IS NULL THEN TRUE ELSE FALSE END AS has_missing_specs,
  
  extracted_at
FROM silver.datasheets_clean

In [0]:
df = spark.table(f"{catalog}.gold.chip_catalog")
display(df)